# 03 - Feature Engineering

Builds per-day features using `src/feature_extraction.py`:
peak load, average load, threshold crossings, crossing ratio, dominant location,
and repetition frequency (how often the same location has crossed the normal-load
line in the recent past) - the features the thesis describes the AI model as
using to recognize a patient's repeated routine (section 3.3).

In [ ]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from feature_extraction import extract_window_features, add_repetition_frequency

intervals = pd.read_csv("../data/intervals.csv")
features = extract_window_features(intervals, window="1D",
                                    normal_limit_kg=8.0, high_risk_limit_kg=12.0)
features = add_repetition_frequency(features, lookback_windows=7)
features.head()


In [ ]:
# Label: Risk if this day's routine crossed the high-risk limit OR the same
# routine has now repeated at least 3 times recently (thesis section 3.3 -
# repeated medium load over time is a risk even without a single high reading).
features["risk_label"] = (
    (features["high_risk_crossings"] > 0) | (features["repetition_frequency"] >= 3)
).astype(int)

print(features["risk_label"].value_counts())
features.to_csv("../data/features.csv", index=False)
print("Saved ../data/features.csv")
